# Model Comparison
Now that `src/preprocessing.py` gives every notebook the same final, cleaned, encoded dataset, this notebook tries a broad sweep of regression models on it — plain linear regression first, then working through regularized linear models, instance-based/kernel methods, and tree ensembles. Everything is scored on the same validation split and the same `KFold` used everywhere else in this project, so the numbers below are directly comparable to `02_baseline_model.ipynb`'s XGBoost baseline (Validation R2 ≈ 0.45).

### Imports
`sys.path` setup to reach `src/` (same reason as `02_baseline_model.ipynb`), the shared pipeline (`get_processed_data`, `get_cv_splitter`, `scale_features`), the models in this sweep, and the same R2/RMSE metrics used elsewhere.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, root_mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from src.preprocessing import get_processed_data, get_cv_splitter, scale_features

### Get the shared dataset
Same call every other notebook uses — one final, cleaned, encoded dataset and one CV splitter, so every model in this sweep is compared on identical footing.

In [2]:
data = get_processed_data()
kfold = get_cv_splitter()

X_train_scaled, X_val_scaled, X_test_scaled = scale_features(data.X_train, data.X_val, data.X_test)

**Why a scaled copy too:** the ordinal-encoded categorical block ranges 0–47 while the binary block is 0/1. Tree models split one column at a time and don't care about magnitude, so they use `data.X_train` directly. Linear/regularized/KNN/SVR models are magnitude-sensitive, so they use `X_train_scaled` (`StandardScaler` fit on `X_train` only, via `src.preprocessing.scale_features`).

**Note on the categorical encoding for linear models:** these models will treat the ordinal category codes as if they carry real numeric magnitude (category 5 vs 10 isn't "twice as much" of anything) — trees don't have this issue, linear models do. Kept as one shared encoding rather than a second one-hot path so every model compares against the exact same dataset; worth remembering if linear/regularized models underperform below for what looks like an odd reason.

### Results table
Every model below gets evaluated the same way: fit → score on train (overfitting check) →
validation R2/RMSE → 5-fold CV R2 (mean ± std) using the shared `kfold`. `gap` is `train_r2 -
val_r2` — a large gap means the model is overfitting (memorizing training rows rather than
generalizing), same diagnostic `02_baseline_model.ipynb` runs for its baseline, now applied to
every model here so it's a direct, comparable check across every model, not just the baseline.
`evaluate()` runs that whole routine once per model and appends a row to `results` so nothing has
to be repeated by hand.

In [3]:
results = []

def evaluate(name, model, X_train, X_val):
    model.fit(X_train, data.y_train)
    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    train_r2 = r2_score(data.y_train, train_pred)
    val_r2 = r2_score(data.y_val, val_pred)
    val_rmse = root_mean_squared_error(data.y_val, val_pred)
    gap = train_r2 - val_r2
    cv_scores = cross_val_score(model, X_train, data.y_train, cv=kfold, scoring='r2')

    results.append({
        'model': name,
        'train_r2': train_r2,
        'val_r2': val_r2,
        'val_rmse': val_rmse,
        'gap': gap,
        'cv_r2_mean': cv_scores.mean(),
        'cv_r2_std': cv_scores.std(),
    })
    print(f"{name}: Train R2={train_r2:.4f}  Val R2={val_r2:.4f}  Gap={gap:.4f}  "
          f"Val RMSE={val_rmse:.4f}  CV R2={cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    return model

## Linear Regression
Already established as the baseline in `02_baseline_model.ipynb` (Val R2=0.5131, RMSE=8.74) — not
duplicated here for its own sake. That notebook only computes a single-split validation R2/RMSE for
it; this one additionally needs its 5-fold CV R2 to place it correctly in the ranked comparison
table below, alongside every other model. Uses the **same unscaled** `data.X_train`/`data.X_val` as
`02_baseline_model.ipynb` (not the scaled copy used for Ridge/Lasso/KNN/SVR below), since scaling is
a mathematical no-op for plain OLS — so the Val R2 here should come out identical to notebook 02's,
not a second, slightly different number.

In [4]:
evaluate('LinearRegression', LinearRegression(), data.X_train, data.X_val)

LinearRegression: Train R2=0.6324  Val R2=0.5131  Gap=0.1193  Val RMSE=8.7404  CV R2=0.5444 ± 0.0219


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](310,)","[ 0.06,-0.04,-0.03,..., 1. ,14.96, 0.42]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](310,)","['cat__X0','cat__X1','cat__X2',...,'num__X380','num__X383','num__X384']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,113
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,310
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(251)


## Regularized linear models
Ridge (L2), Lasso (L1), ElasticNet (L1+L2) — same "shrink the coefficients" idea as Linear Regression, different penalty shape on how the shrinkage is applied. Default hyperparameters for now, as a fair before-tuning comparison point.

In [5]:
evaluate('Ridge', Ridge(random_state=42), X_train_scaled, X_val_scaled)
evaluate('Lasso', Lasso(random_state=42), X_train_scaled, X_val_scaled)
evaluate('ElasticNet', ElasticNet(random_state=42), X_train_scaled, X_val_scaled)

Ridge: Train R2=0.6324  Val R2=0.5131  Gap=0.1193  Val RMSE=8.7410  CV R2=0.5473 ± 0.0223


Lasso: Train R2=0.5533  Val R2=0.5355  Gap=0.0178  Val RMSE=8.5373  CV R2=0.5499 ± 0.0276
ElasticNet: Train R2=0.5612  Val R2=0.5407  Gap=0.0204  Val RMSE=8.4890  CV R2=0.5494 ± 0.0282


,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"alpha alpha: float, default=1.0Constant that multiplies the penalty terms. Defaults to 1.0.See the notes for the exact mathematical meaning of thisparameter. ``alpha = 0`` is equivalent to an ordinary least square,solved by the :class:`LinearRegression` object. For numericalreasons, using ``alpha = 0`` with the ``Lasso`` object is not advised.Given this, you should use the :class:`LinearRegression` object.",1.0
,"l1_ratio l1_ratio: float, default=0.5The ElasticNet mixing parameter, with ``0 <= l1_ratio <= 1``. For``l1_ratio = 0`` the penalty is an L2 penalty. ``For l1_ratio = 1`` itis an L1 penalty. For ``0 < l1_ratio < 1``, the penalty is acombination of L1 and L2.",0.5
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If ``False``, thedata is assumed to be already centered.",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.Check :ref:`an example on how to use a precomputed Gram Matrix in ElasticNet<sphx_glr_auto_examples_linear_model_plot_elastic_net_precomputed_gram_matrix_with_weighted_samples.py>`for details.",False
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary <warm_start>`.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'


## Instance-based & kernel models
KNN and SVR — both distance/kernel-based, so both need the scaled features. These tend to struggle on wide, high-dimensional, mostly-binary data like this, but included for a complete comparison.

In [6]:
evaluate('KNeighborsRegressor', KNeighborsRegressor(), X_train_scaled, X_val_scaled)
evaluate('SVR', SVR(), X_train_scaled, X_val_scaled)

KNeighborsRegressor: Train R2=0.6103  Val R2=0.3953  Gap=0.2150  Val RMSE=9.7410  CV R2=0.4122 ± 0.0195


SVR: Train R2=0.4885  Val R2=0.4562  Gap=0.0322  Val RMSE=9.2370  CV R2=0.4457 ± 0.0267


,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",0.1
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


## Decision Tree
A single decision tree, unscaled features, default hyperparameters (no depth limit) — the building
block `RandomForest`/`XGBoost`/`LightGBM`/`CatBoost` below are all *ensembles of*. With no limit on
how deep it can grow, it will keep splitting until it separates almost every training row into its
own leaf — a strong candidate for the most dramatic overfitting in this whole comparison, worth
watching for in the CV R² below.

In [7]:
evaluate('DecisionTreeRegressor', DecisionTreeRegressor(random_state=42), data.X_train, data.X_val)

DecisionTreeRegressor: Train R2=0.9751  Val R2=0.1484  Gap=0.8266  Val RMSE=11.5593  CV R2=0.1691 ± 0.0958


,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes 

## Tree ensembles
RandomForest, XGBoost, LightGBM, CatBoost — unscaled features (trees don't need scaling), default hyperparameters, `random_state=42` for reproducibility.

In [8]:
evaluate('RandomForestRegressor', RandomForestRegressor(random_state=42), data.X_train, data.X_val)
evaluate('XGBRegressor', XGBRegressor(random_state=42), data.X_train, data.X_val)
evaluate('LGBMRegressor', LGBMRegressor(random_state=42, verbose=-1), data.X_train, data.X_val)
evaluate('CatBoostRegressor', CatBoostRegressor(random_state=42, verbose=0), data.X_train, data.X_val)

RandomForestRegressor: Train R2=0.9177  Val R2=0.4951  Gap=0.4226  Val RMSE=8.9010  CV R2=0.5376 ± 0.0330


XGBRegressor: Train R2=0.8839  Val R2=0.4801  Gap=0.4038  Val RMSE=9.0319  CV R2=0.5035 ± 0.0426


LGBMRegressor: Train R2=0.7701  Val R2=0.5484  Gap=0.2217  Val RMSE=8.4183  CV R2=0.5694 ± 0.0252


CatBoostRegressor: Train R2=0.7852  Val R2=0.5332  Gap=0.2520  Val RMSE=8.5583  CV R2=0.5728 ± 0.0238


CatBoostRegressor(loss_function='RMSE', random_state=42, verbose=0)

## Comparison
Sorted by CV R2 (mean across the 5 folds) — the more reliable number since it's not sensitive to which particular rows landed in `X_val`. `val_r2`/`val_rmse` are kept alongside for a direct comparison against `02_baseline_model.ipynb`'s numbers, and `gap` (`train_r2 - val_r2`) shows how much each model is overfitting — worth reading alongside `cv_r2_std`, since an unstable model (high CV std) and an overfitting model (high gap) are related but not identical symptoms.

In [9]:
results_df = pd.DataFrame(results).sort_values('cv_r2_mean', ascending=False).reset_index(drop=True)
results_df

,model,train_r2,val_r2,val_rmse,gap,cv_r2_mean,cv_r2_std
0,CatBoostRegressor,0.785227,0.533207,8.558331,0.252020,0.572768,0.023791
1,LGBMRegressor,0.770077,0.548358,8.418295,0.221719,0.569388,0.025161
2,Lasso,0.553301,0.535495,8.537334,0.017806,0.549936,0.027610
3,ElasticNet,0.561189,0.540740,8.488999,0.020449,0.549352,0.028206
4,Ridge,0.632384,0.513071,8.740970,0.119312,0.547344,0.022312
5,LinearRegression,0.632398,0.513131,8.740438,0.119267,0.544367,0.021902
6,RandomForestRegressor,0.917675,0.495082,8.900974,0.422594,0.537624,0.032952
7,XGBRegressor,0.883893,0.480113,9.031949,0.403780,0.503478,0.042552
8,SVR,0.488453,0.456237,9.237023,0.032216,0.445723,0.026673
9,KNeighborsRegressor,0.610259,0.395277,9.741041,0.214982,0.412151,0.019513
